In [5]:
from itext2kg import itext2kg_star
from langchain_ollama import ChatOllama, OllamaEmbeddings

In [15]:
import time
from datetime import datetime

In [7]:
llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0,
)

In [11]:
llm_text = ChatOllama(
    model="gemma2:2b",
    temperature=0,
)
embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest",
)

In [9]:
from langchain.document_loaders import PyPDFLoader
loader = PyPDFLoader(f"./data/input/2020-Scrum-Guide-US.pdf")
pages = loader.load_and_split()

In [13]:
from itext2kg.documents_distiller import DocumentsDistiller, Article
document_distiller = DocumentsDistiller(llm_model = llm)

In [21]:
# Record start time
start_time = time.time()
start_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Process started at: {start_timestamp}")

IE_query = '''
# DIRECTIVES : 
- Act like an experienced information extractor. 
- You have a document describing the Scrum software development process.
- If you do not find the right information, keep its place empty.
'''
# we have replaced the curly braces with square brackets to avoid the error in the query
distilled_text = await document_distiller.distill(documents=[page.page_content.replace("{", '[').replace("}", "]") for page in pages], IE_query=IE_query, output_data_structure=Article)

end_time = time.time()
end_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
elapsed_time = end_time - start_time

print(f"Process finished at: {end_timestamp}")
print(f"Time elapsed: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")

Process started at: 2025-08-20 09:50:24
Process finished at: 2025-08-20 09:52:28
Time elapsed: 123.76 seconds (2.06 minutes)


In [23]:
distilled_text

Article(title="Scrum Guide Purpose of the Scrum Guide Purpose of the Scrum Guide Definition of Done Scrum Software Development Process Scrum Software Development Process Scrum Software Development Process Scrum Master's Role and Responsibilities Scrum Software Development Process Scrum Software Development Process Scrum Software Development Process Scrum Software Development Process # Context: 11 Definition of Done Scrum Software Development Process", authors=[Author(name='Ken Schwaber', affiliation='Scrum.org'), Author(name='Jeff Sutherland', affiliation='Scrum.org'), Author(name='Ken Schwaber', affiliation='Scrum Founder'), Author(name='Jeff Sutherland', affiliation='Scrum Co-Founder'), Author(name='Scrum Guide', affiliation='Scrum.org'), Author(name='Unknown', affiliation=''), Author(name='Scrum Team', affiliation='Scrum Team'), Author(name='Stakeholders', affiliation='Stakeholders'), Author(name='Scrum Team', affiliation='5 Scrum Team'), Author(name='Product Owner', affiliation='5 

In [33]:
semantic_blocks = [f"{key} - {value}".replace("{", "[").replace("}", "]") 
                     for key, value in distilled_text.model_dump().items() 
                     if value != [] and value != "" and value is not None]

In [35]:
import asyncio
from itext2kg import iText2KG_Star
from itext2kg.logging_config import setup_logging, get_logger

# Initialize iText2KG_Star with the llm model and embeddings model
itext2kg_star = iText2KG_Star(llm_model=llm_text, embeddings_model=embeddings)

# Your text sections (can be facts from document distiller or raw text)
sections=semantic_blocks

logger.info("Starting knowledge graph construction with iText2KG_Star...")

# Build the knowledge graph - entities are automatically derived from relationships
kg = await itext2kg_star.build_graph(
    sections=sections,
    ent_threshold=0.8,      # Higher threshold for more distinct entities
    rel_threshold=0.7,      # Threshold for relationship merging
    observation_date="2025-01-15"  # Optional: add temporal context
)

logger.info(f"Knowledge graph completed! Entities: {len(kg.entities)}, Relationships: {len(kg.relationships)}")

[2025-08-20 09:56:36] [    INFO] [itext2kg.__main__] Starting knowledge graph construction with iText2KG_Star...
[2025-08-20 09:56:36] [    INFO] [itext2kg.itext2kg.itext2kg_star] ------- Extracting Relations and Deriving Entities from Document 1
[2025-08-20 09:57:06] [    INFO] [itext2kg.itext2kg.itext2kg_star] ------- Extracting Relations and Deriving Entities from Document 2
[2025-08-20 09:58:01] [    INFO] [itext2kg.itext2kg.graph_matching.matcher] Entity was matched --- [scrum guide:authors] --merged--> [scrum guide:title]
[2025-08-20 09:58:01] [    INFO] [itext2kg.itext2kg.graph_matching.matcher] Entity was matched --- [scrum software development process:affiliation] --merged--> [scrum software development process:Scrum_Master_s_Role_and_Responsibilities]
[2025-08-20 09:58:01] [    INFO] [itext2kg.itext2kg.itext2kg_star] ------- Extracting Relations and Deriving Entities from Document 3
[2025-08-20 09:58:36] [    INFO] [itext2kg.itext2kg.graph_matching.matcher] Entity was matched